In [18]:
import math
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

DATA_PATH = "Q-RMR Data.xlsx"
FEATURE_COLS = ["RQD", "Jn", "Jr", "Ja", "Jw", "SRF"]
TARGET_COL = "RMR"
R2_COLS = ["R2_train", "R2_val", "R2_test"]

MODELS = []

def register(name, family, ctor_fn):
    MODELS.append((name, family, ctor_fn))

def load_and_split(seed):
    df = pd.read_excel(DATA_PATH)
    df = df[FEATURE_COLS + [TARGET_COL]].copy()
    df[FEATURE_COLS + [TARGET_COL]] = df[FEATURE_COLS + [TARGET_COL]].apply(pd.to_numeric, errors="coerce")
    df = df.dropna().reset_index(drop=True)

    X = df[FEATURE_COLS].values
    y = df[TARGET_COL].values

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=seed, shuffle=True
    )
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, shuffle=True
    )

    return X_tr, X_val, X_te, y_tr, y_val, y_te, len(y)

def metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

def style_df(df):
    sty = df.style.format(precision=4, na_rep="-")
    if any(c in df.columns for c in R2_COLS):
        sty = sty.set_properties(
            subset=[c for c in R2_COLS if c in df.columns],
            **{"color": "red", "font-weight": "bold"}
        )
    return sty

def evaluate_and_show(seed):
    X_tr, X_val, X_te, y_tr, y_val, y_te, n = load_and_split(seed)

    rows = []
    for name, family, ctor in MODELS:
        try:
            model = ctor()
            model.fit(X_tr, y_tr)

            yhat_tr = np.asarray(model.predict(X_tr)).reshape(-1)
            yhat_val = np.asarray(model.predict(X_val)).reshape(-1)
            yhat_te = np.asarray(model.predict(X_te)).reshape(-1)

            r2_tr, rmse_tr, mae_tr = metrics(y_tr, yhat_tr)
            r2_val, rmse_val, mae_val = metrics(y_val, yhat_val)
            r2_te, rmse_te, mae_te = metrics(y_te, yhat_te)

            rows.append([
                name, family, "ok",
                r2_tr, rmse_tr, mae_tr,
                r2_val, rmse_val, mae_val,
                r2_te, rmse_te, mae_te
            ])
        except Exception as e:
            rows.append([name, family, f"fail: {str(e)[:80]}"] + [np.nan] * 9)

    metrics_df = pd.DataFrame(rows, columns=[
        "Model", "Family", "Status",
        "R2_train", "RMSE_train", "MAE_train",
        "R2_val", "RMSE_val", "MAE_val",
        "R2_test", "RMSE_test", "MAE_test"
    ]).sort_values(by=["R2_val", "R2_test"], ascending=[False, False]).reset_index(drop=True)

    r2_only_df = metrics_df[["Model", "Family", "Status"] + R2_COLS].copy()

    print(f"Data = '{DATA_PATH}' | N = {n} | Seed = {seed} | Models = {len(MODELS)}")
    print("Full model comparison (sorted by validation R²)")
    display(style_df(metrics_df))

    print("R² only (sorted by validation R²)")
    display(style_df(r2_only_df))

    return metrics_df, r2_only_df

In [20]:
# XGBoost

import xgboost as xgb

def build_xgb():
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        learning_rate=0.1,
        max_depth=5,
        n_estimators=60,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        subsample=1.0,
        tree_method="hist",
        verbosity=0,
        random_state=437
    )

register("XGBoost", "Boosting", build_xgb)

In [22]:
# GBDT

from sklearn.ensemble import HistGradientBoostingRegressor

def build_gbdt():
    return HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=100,
        max_depth=None,
        max_features=1.0,
        learning_rate=0.2,
        min_samples_leaf=10,
        l2_regularization=0.0,
        categorical_features="from_dtype",
        warm_start=False,
        early_stopping="auto",
        scoring="loss",
        validation_fraction=0.1,
        n_iter_no_change=10,
        tol=1e-7,
        verbose=0,
        random_state=437
    )

register("GBDT", "Boosting", build_gbdt)

In [24]:
# RF

from sklearn.ensemble import RandomForestRegressor

def build_rf():
    return RandomForestRegressor(
        criterion="squared_error",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=1.0,
        n_estimators=200,
        bootstrap=True,
        oob_score=False,
        n_jobs=1,
        verbose=0,
        warm_start=False,
        ccp_alpha=0.0,
        max_samples=None,
        monotonic_cst=None,
        random_state=437
    )

register("RF", "Bagging", build_rf)

In [26]:
# DT

from sklearn.tree import DecisionTreeRegressor

def build_dt():
    return DecisionTreeRegressor(
        criterion="squared_error",
        splitter="best",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features=None,
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        ccp_alpha=0.0,
        monotonic_cst=None,
        random_state=437
    )

register("DT", "Tree", build_dt)

In [28]:
# KNN

from sklearn.neighbors import KNeighborsRegressor

def build_knn():
    return KNeighborsRegressor(
        n_neighbors=5,
        weights="uniform",
        algorithm="auto",
        leaf_size=30,
        p=2,
        metric="minkowski",
        metric_params=None,
        n_jobs=1
    )

register("KNN", "Neighbors", build_knn)

In [30]:
#GPR

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

def build_gpr():
    kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)
    return GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.0,
        optimizer="fmin_l_bfgs_b",
        n_restarts_optimizer=2,
        normalize_y=False,
        copy_X_train=True,
        n_targets=None,
        random_state=437
    )

register("GPR", "Kernel", build_gpr)

In [32]:
SEED = 0
metrics_df, r2_only_df = evaluate_and_show(SEED)

Data = 'Q-RMR Data.xlsx' | N = 448 | Seed = 0 | Models = 6
Full model comparison (sorted by validation R²)


,Model,Family,Status,R2_train,RMSE_train,MAE_train,R2_val,RMSE_val,MAE_val,R2_test,RMSE_test,MAE_test
0,XGBoost,Boosting,ok,0.9768,1.0019,0.6340,0.9225,2.0123,1.0679,0.9008,2.4110,1.1287
1,GBDT,Boosting,ok,0.9744,1.0528,0.5392,0.9185,2.0634,1.0677,0.8644,2.8186,1.1624
2,RF,Bagging,ok,0.9492,1.4827,0.6268,0.8978,2.3106,1.0817,0.8775,2.6788,1.0197
3,GPR,Kernel,ok,0.9951,0.4589,0.2075,0.8577,2.7267,1.2152,0.8048,3.3819,1.0749
4,KNN,Neighbors,ok,0.7696,3.1586,1.6992,0.7452,3.6491,1.8116,0.7125,4.1041,1.9225
5,DT,Tree,ok,0.9202,1.8590,0.7803,0.6898,4.0258,1.8447,0.6872,4.2804,1.6091


R² only (sorted by validation R²)


,Model,Family,Status,R2_train,R2_val,R2_test
0,XGBoost,Boosting,ok,0.9768,0.9225,0.9008
1,GBDT,Boosting,ok,0.9744,0.9185,0.8644
2,RF,Bagging,ok,0.9492,0.8978,0.8775
3,GPR,Kernel,ok,0.9951,0.8577,0.8048
4,KNN,Neighbors,ok,0.7696,0.7452,0.7125
5,DT,Tree,ok,0.9202,0.6898,0.6872
